# 4.4 Development of a Stochastic Simulation Model

### Model Stucture
The daily cargo throughput is modeled as Compound Poisson Process (CPP) for each vessel type (Container, Tanker, Bulk).

Let $N_v(t)$ denote the number of vessel arrivals for category $v \in \{C, T, B\}$ over the time interval $[0, t]$. The total cargo handeled by all vessels of type $v$ in this interval is defined as the sum:

$S_{v} = X_{v, 1} + X_{v, 2} + ... + X_{v, N_v(t)}$

Where: 
- $N_v(t) \sim \text{Poisson}(\lambda_v t)$ is a homogeneous Poisson process representing the arrival of vessels
- $X_{v, i} \sim \text{Lognormal}(\mu_v, \sigma_v^2)$ is the cargo volume of the i-th vessel of type v


To formally relate these variables, the expected value and variance of the daily throughput for a given vessel type are defined as:
   - $E[S_v(t)] = \lambda_v t \cdot E[X_{v,i}]$
   - $\text{Var}[S_v(t)] = \lambda_v t \cdot E[X_{v,i}^2]$

The total cargo throughput for the port over time $t$ is the sum of these three independent Poisson processes, which (according to $\text{Theorem 3.2.1.}$ from the Lecture Notes) is a CPP with rate $\lambda = \lambda_C + \lambda_T + \lambda_B$: 

$S(t) = S_C(t) + S_T(t) + S_B(t)$

### Model assumptions
The validity of this compound Poisson model relies on the following assumptions:

- Independent Arrivals: Vessel arrivals are mutually independent and occur with stationary increments (2 core properties of the homogeneous Poisson process).
- Independent Cargo Sizes: The cargo volumes $X_{v, i}$ (for each i) are mutually independent and occur with stationary increments (properties of the Poisson process), and are also independent of the number of arriving vessels $N_v(t)$ (property of the CPP).
- Identical Distributions: The cargo volumes $X_{v, i}$ within a specific vessel category are identically distributed (another property of the CPP).
- Category Independence: The arrivals and volumes of Containers, Tankers and Bulk vessels do not influence one another (another property of the CPP).

### Parameter estimation methods
To parametrize this scenario, the Method of Moments is applied using the provided data:

- Arrival Rates ($\lambda_v$): The expected number of daily arrivals $E[N_v(1)]$ is estimated using the sample mean of daily vessel counts for each category. 
- Cargo Volumes ($\mu_v, \sigma_v^2$): We calculate the first moment $M_1 = \frac{1}{n}\sum X_i$ (sample mean) and the second moment $M_2 = \frac{1}{n}\sum X_i^2$ (sample mean of squares). From these, the Lognormal parameters are derived as:

$\sigma_v^2 = \ln\left(\frac{M_2}{M_1^2}\right)$,
$\mu_v = \ln(M_1) - \frac{1}{2}\sigma_v^2$

These calculations for $\lambda_v$, $\mu_v$ and $\sigma_v^2$ have been done previously in tasks 4.2 and 4.3 respectively:

In [ ]:
import pandas as pd

In [6]:
df = (
    pd.read_excel("rotterdam_dataset_15.xlsx")
      .sort_values("Arrival_Date")
      .reset_index(drop=True)
)
df['Vessel_Type'] = df['Vessel_Type'].astype('category')
df['Arrival_Day'] = df['Arrival_Date'].dt.date

In [7]:
d = df['Arrival_Date']
n_days = d.dt.normalize().nunique()

arr_daily = len(df) / n_days
arr_daily_per_type = df.groupby("Vessel_Type", observed=True).size() / n_days

print(arr_daily)
print(arr_daily_per_type)

15.584854014598541
Vessel_Type
Bulk         2.406934
Container    8.664234
Tanker       4.513686
dtype: float64


So the arrival rates for each type of vessel are as follows:
- $\lambda_C = 8.664234$
- $\lambda_T = 4.513686$
- $\lambda_B = 2.406934$

And the arrival rate of the CPP is:

$\lambda = \lambda_C + \lambda_T + \lambda_B = 8.664234 + 4.513686 + 2.406934 = 15.584854014598541$

In [9]:
import numpy as np

In [ ]:
for t in ["Container", "Tanker", "Bulk"]:
	cargo = df[df["Vessel_Type"] == t]["Cargo_tons"].values

	M1 = np.mean(cargo)
	M2 = np.mean(np.power(cargo, 2))

	sigma2 = np.log(M2 / (M1 ** 2))
	mu = np.log(M1) - (sigma2 / 2)
	sigma = np.sqrt(sigma2)

	print(f"{t}: mu = {mu:.4f}, sigma = {sigma:.4f}")

Container: mu = 9.5498, sigma = 0.3433
Tanker: mu = 10.6743, sigma = 0.3014
Bulk: mu = 10.0524, sigma = 0.3929


So the data yields the following parameters:
- $\mu_C = 9.5498, \sigma_C = 0.3433$
- $\mu_T = 10.6743, \sigma_T = 0.3014$
- $\mu_B = 10.0524, \sigma_B = 0.3929$

### Model Justification
The Compound Poisson Process is the best mathematical model for this problem because daily port traffic is driven by two random factors: how many ships arrive, and how much cargo each ship carries.

When looking at different options to model the cargo volumes ($X_{v,i}$), we chose the Lognormal distribution over the Normal and Gamma distributions based on our data analysis.
The cargo data is always greater than zero and has a "long tail" on the right side (meaning most ships carry an average amount, but a few carry massive amounts). We rejected the Normal distribution because its symmetrical shape means it would predict impossible negative cargo weights. While the Gamma distribution fixes the negative weight problem, we still rejected it because our statistical checks (specifically the Kolmogorov-Smirnov test) showed that it simply didn't match the data as well as the Lognormal option.Ultimately, the Lognormal distribution is the best choice. It only produces positive numbers, and it does a great job of capturing the rare but big volumes that are most likely to cause severe port congestion.